# 02 Handmade Augmentation and Replication

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP_CMB_003_KNN_Combined_Handmade_Augmentation.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/scripts/EXP_CMB_003_KNN_Combined_Handmade_Augmentation.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/logs)
- [`EXP_CMB_004_KNN_Combined_Sample_Influence.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/scripts/EXP_CMB_004_KNN_Combined_Sample_Influence.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/KNN/Combined/scripts/EXP_CMB_003_KNN_Combined_Handmade_Augmentation.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '379054719ee866d414ce3cc858158e80c30ef2e1'}, '05_model_optimize/KNN/Combined/scripts/EXP_CMB_004_KNN_Combined_Sample_Influence.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '1b8445dbbfb043a10f050f6352d12647f640c82c'}}

## EXP_CMB_003_KNN_Combined_Handmade_Augmentation

원본: [05_model_optimize/KNN/Combined/scripts/EXP_CMB_003_KNN_Combined_Handmade_Augmentation.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/scripts/EXP_CMB_003_KNN_Combined_Handmade_Augmentation.py)  
Git blob: `379054719ee866d414ce3cc858158e80c30ef2e1`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/logs)

EXP-CMB-003: writer-balanced, training-only Hand-made augmentation.

Place in: 05_model_optimize/KNN/Combined/scripts/
Run from repository root:
 python 05_model_optimize/KNN/Combined/scripts/EXP_CMB_003_KNN_Combined_Handmade_Augmentation.py

Frozen optimized Normal source: 76,800 official Training + 30,720 official-derived
augmentation rows (107,520 total). Each of four writer-held-out folds adds
1,920 original samples from the other three writers. Control vs +480/+960/+1920
size/stroke-augment



In [ ]:
%%archive_source 05_model_optimize/KNN/Combined/scripts/EXP_CMB_003_KNN_Combined_Handmade_Augmentation.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-CMB-003: writer-balanced, training-only Hand-made augmentation.

Place in: 05_model_optimize/KNN/Combined/scripts/
Run from repository root:
 python 05_model_optimize/KNN/Combined/scripts/EXP_CMB_003_KNN_Combined_Handmade_Augmentation.py

Frozen optimized Normal source: 76,800 official Training + 30,720 official-derived
augmentation rows (107,520 total). Each of four writer-held-out folds adds
1,920 original samples from the other three writers. Control vs +480/+960/+1920
size/stroke-augmented Hand-made samples, respectively. Never loads official Test.
The 4-writer OOF is development-only: Normal features and targeted classes were
previously selected using these writers' development predictions.
"""
from __future__ import annotations

import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn

ROOT = Path(__file__).resolve().parents[4]
NORMAL_DIR = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
COMBINED_DIR = ROOT / '05_model_optimize' / 'KNN' / 'Combined' / 'scripts'
for p in (NORMAL_DIR, COMBINED_DIR, ROOT / '03_experiments' / 'model_optimization'):
    sys.path.insert(0, str(p))

from common import LABELS, fixed_split
from EXP004_KNN_Normal_CoM import BatchedCoM
from EXP007_KNN_Normal_Augmentation import size_jitter, stroke_jitter
from final_knn_normal import assemble_training, prepare_features, EXPECTED_VALIDATION_CORRECT
from EXP_CMB_001_KNN_Combined_Baseline import WRITERS, load_writer, save_csv, save_cm, summary
from EXP_CMB_002_KNN_Combined_Optimized import fresh_knn, paired_counts

OUT = ROOT / '05_model_optimize' / 'KNN' / 'Combined' / 'results' / 'EXP_CMB_003_handmade_augmentation'
SEED = 20261123
AUG_RATIOS = (0, 25, 50, 100)
AUG_STRENGTH = 0.44
EXPECTED_CONTROL_WRITER_CORRECT = {41: 586, 42: 595, 43: 619, 44: 594}
EXPECTED_CONTROL_TOTAL = 2394
# Guardrail defined before evaluating the new variants, not changed afterward.
MIN_OFFICIAL_VALIDATION_ACCURACY = 0.978


def balanced_nested_indices(writer_ids, labels, seed):
    """Within each of 3 writers x 16 classes, deterministic 10/20/40 of 40.

    Ensures +25%, +50%, +100% are nested and no held-out writer is sampled.
    """
    rng = np.random.default_rng(seed)
    parts = []
    for w in sorted(set(int(x) for x in writer_ids)):
        for c in range(16):
            idx = np.flatnonzero((writer_ids == w) & (labels == c))
            if len(idx) != 40:
                raise RuntimeError(f'Expected 40 samples for writer {w}, class {LABELS[c]}')
            parts.append(rng.permutation(idx))
    return {
        25: np.sort(np.concatenate([p[:10] for p in parts])),
        50: np.sort(np.concatenate([p[:20] for p in parts])),
        100: np.sort(np.concatenate(parts)),
    }


def make_aug_features(raw, selected, com, ratio, heldout):
    if len(selected) != (1920 * ratio // 100):
        raise RuntimeError(f'Invalid augmentation count for {ratio}%')
    resized, size_stats = size_jitter(raw, selected, seed=SEED + heldout + 100)
    strokes, stroke_stats = stroke_jitter(resized, strength=AUG_STRENGTH)
    del resized
    features = prepare_features(strokes, com, f'handmade augmented heldout={heldout} ratio={ratio}%')
    del strokes
    return features, {'size_counts': size_stats, 'stroke': stroke_stats}


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    xtr, ytr, xva, yva, official_sha = fixed_split()
    if (len(ytr), len(yva)) != (76800, 19200):
        raise RuntimeError('Official split mismatch; refusing experiment')
    handmade = {w: load_writer(w) for w in WRITERS}
    com = BatchedCoM()
    print('Official Test NOT loaded. Preparing frozen optimized Normal features...', flush=True)
    official_x, official_y = assemble_training(xtr, ytr, com)
    del xtr, ytr
    if official_x.shape != (107520, 1108):
        raise RuntimeError('Frozen Normal train feature shape mismatch')
    official_val_x = prepare_features(xva, com, 'official Validation')
    del xva
    original_features = {w: prepare_features(handmade[w][0], com, f'writer {w}') for w in WRITERS}

    print('\n[Normal reference] Checking official Validation and 4-writer development data...', flush=True)
    normal = fresh_knn()
    normal.fit(official_x, official_y)
    reference_val = summary(yva, normal.predict(official_val_x).astype(np.int64))
    if reference_val['correct'] != EXPECTED_VALIDATION_CORRECT:
        raise RuntimeError('Official Validation frozen Normal reproduction failed')
    normal_by_writer = {w: normal.predict(original_features[w]).astype(np.int64) for w in WRITERS}
    normal_truth = np.concatenate([handmade[w][1] for w in WRITERS])
    normal_oof = np.concatenate([normal_by_writer[w] for w in WRITERS])
    normal_result = summary(normal_truth, normal_oof)
    if normal_result['correct'] != 2381:
        raise RuntimeError('Frozen optimized Normal handwriting reference mismatch')
    print('Frozen Normal EXACTLY reproduced: official=%0.6f, handmade=%0.6f' %
          (reference_val['accuracy'], normal_result['accuracy']), flush=True)
    del normal

    variants = {r: {} for r in AUG_RATIOS}
    metrics_by_fold = []
    official_rows = []
    augmentation_records = []
    for heldout in WRITERS:
        source_writers = [w for w in WRITERS if w != heldout]
        raw_source = np.concatenate([handmade[w][0] for w in source_writers], axis=0)
        raw_labels = np.concatenate([handmade[w][1] for w in source_writers])
        ids = np.concatenate([np.full(640, w, dtype=np.int32) for w in source_writers])
        original_x = np.concatenate([original_features[w] for w in source_writers])
        if original_x.shape != (1920, 1108) or set(np.unique(ids)) != set(source_writers):
            raise RuntimeError('Wrong training writers or source feature shape')
        selected = balanced_nested_indices(ids, raw_labels, seed=SEED + heldout)
        if not (set(selected[25]) <= set(selected[50]) <= set(selected[100])):
            raise RuntimeError('Augmentation indices not nested')
        print(f'\n[Fold heldout={heldout}] training writers={source_writers}, base=109440 rows', flush=True)
        for ratio in AUG_RATIOS:
            if ratio:
                new_x, aug_stats = make_aug_features(raw_source, selected[ratio], com, ratio, heldout)
                extra_y = raw_labels[selected[ratio]]
                fit_x = np.concatenate([official_x, original_x, new_x], axis=0)
                fit_y = np.concatenate([official_y, raw_labels, extra_y])
                del new_x
                augmentation_records.append({'heldout_writer': heldout, 'ratio': ratio,
                                             'source_writers': source_writers, **aug_stats})
            else:
                fit_x = np.concatenate([official_x, original_x], axis=0)
                fit_y = np.concatenate([official_y, raw_labels])
            expected_n = 109440 + (1920 * ratio // 100)
            if fit_x.shape != (expected_n, 1108) or fit_y.shape != (expected_n,):
                raise RuntimeError('Fit data dimension mismatch')
            model = fresh_knn()
            start = time.perf_counter()
            model.fit(fit_x, fit_y)
            fit_s = time.perf_counter() - start
            del fit_x, fit_y
            start = time.perf_counter()
            pred_hand = model.predict(original_features[heldout]).astype(np.int64)
            hand_s = time.perf_counter() - start
            start = time.perf_counter()
            pred_off = model.predict(official_val_x).astype(np.int64)
            off_s = time.perf_counter() - start
            del model
            hand_stats = summary(handmade[heldout][1], pred_hand)
            off_stats = summary(yva, pred_off)
            if ratio == 0 and hand_stats['correct'] != EXPECTED_CONTROL_WRITER_CORRECT[heldout]:
                raise RuntimeError(f'EXP-CMB-002 control mismatch for writer {heldout}: '
                                   f"{hand_stats['correct']} != {EXPECTED_CONTROL_WRITER_CORRECT[heldout]}")
            variants[ratio][heldout] = pred_hand
            row = {'variant': 'control' if ratio == 0 else f'handmade_aug_{ratio}',
                   'ratio_percent': ratio, 'heldout_writer': heldout,
                   'train_writers': ','.join(map(str, source_writers)),
                   'official_derived_train': 107520, 'original_handmade_train': 1920,
                   'augmented_handmade_train': ratio * 1920 // 100,
                   'total_train': expected_n,
                   'heldout_n': 640, 'handmade_correct': hand_stats['correct'],
                   'handmade_accuracy': hand_stats['accuracy'],
                   'handmade_macro_f1': hand_stats['macro_f1'],
                   'official_accuracy': off_stats['accuracy'],
                   'official_macro_f1': off_stats['macro_f1'],
                   'official_correct': off_stats['correct'],
                   'official_guardrail_pass': int(off_stats['accuracy'] >= MIN_OFFICIAL_VALIDATION_ACCURACY),
                   'fit_s': round(fit_s, 3), 'handmade_predict_s': round(hand_s, 3),
                   'official_predict_s': round(off_s, 3)}
            metrics_by_fold.append(row)
            official_rows.append({k: row[k] for k in ('variant', 'heldout_writer', 'official_correct',
                                                      'official_accuracy', 'official_macro_f1')})
            save_csv(OUT / 'fold_results_partial.csv', metrics_by_fold)
            save_csv(OUT / 'official_validation_partial.csv', official_rows)
            print(f'  ratio={ratio:3d}% train={expected_n}: '
                  f"handmade={hand_stats['accuracy']:.6f} "
                  f"official={off_stats['accuracy']:.6f} "
                  f"guardrail={'PASS' if row['official_guardrail_pass'] else 'FAIL'}", flush=True)
        del raw_source, raw_labels, original_x

    summary_rows = []
    aggregates = {}
    predictions = []
    for ratio in AUG_RATIOS:
        pred = np.concatenate([variants[ratio][w] for w in WRITERS])
        agg = summary(normal_truth, pred)
        paired = paired_counts(normal_truth, np.concatenate([variants[0][w] for w in WRITERS]), pred)
        min_official = min(r['official_accuracy'] for r in metrics_by_fold if r['ratio_percent'] == ratio)
        entry = {'variant': 'control' if ratio == 0 else f'handmade_aug_{ratio}',
                 'ratio_percent': ratio, 'correct': agg['correct'], 'accuracy': agg['accuracy'],
                 'macro_f1': agg['macro_f1'], 'min_fold_official_accuracy': min_official,
                 'official_guardrail_pass': int(min_official >= MIN_OFFICIAL_VALIDATION_ACCURACY),
                 'delta_pp_vs_control': 100 * (agg['accuracy'] - EXPECTED_CONTROL_TOTAL / 2560),
                 'paired_vs_control': paired}
        aggregates[entry['variant']] = {'oof': agg, 'paired_vs_control': paired}
        summary_rows.append({k: v for k, v in entry.items() if k != 'paired_vs_control'})
        save_cm(OUT / f'{entry["variant"]}_oof_confusion_matrix.csv', normal_truth, pred)
        save_csv(OUT / f'{entry["variant"]}_per_class_oof.csv', [
            {'class': lbl, **agg['per_class'][lbl]} for lbl in LABELS
        ])
        for wi, w in enumerate(WRITERS):
            for i in range(640):
                ix = wi * 640 + i
                predictions.append({'variant': entry['variant'], 'writer': w, 'index': i,
                                    'true': LABELS[int(normal_truth[ix])],
                                    'predicted': LABELS[int(pred[ix])],
                                    'correct': int(pred[ix] == normal_truth[ix])})
    if aggregates['control']['oof']['correct'] != EXPECTED_CONTROL_TOTAL:
        raise RuntimeError('EXP-CMB-002 combined OOF control not reproduced')
    eligible = [r for r in summary_rows if r['official_guardrail_pass']]
    if not eligible:
        raise RuntimeError('All variants failed prespecified official Validation guardrail')
    best = max(eligible, key=lambda r: (r['macro_f1'], r['accuracy'], -r['ratio_percent']))
    save_csv(OUT / 'fold_results.csv', metrics_by_fold)
    save_csv(OUT / 'official_validation.csv', official_rows)
    save_csv(OUT / 'candidate_results.csv', summary_rows)
    save_csv(OUT / 'oof_predictions.csv', predictions)
    report = {
        'experiment': 'EXP-CMB-003: 3-writer Hand-made size+stroke augmentation ratios (4-writer development OOF)',
        'official_dataset_sha256': official_sha,
        'handmade_sha256': {str(w): handmade[w][2] for w in WRITERS},
        'official_training_source': 76800,
        'official_derived_training': 107520,
        'handmade_original_per_fold': 1920,
        'handmade_augmented_per_fold': {str(r): 1920 * r // 100 for r in AUG_RATIOS},
        'total_training_per_fold': {str(r): 109440 + (1920*r // 100) for r in AUG_RATIOS},
        'heldout_samples_per_fold': 640,
        'method': {'preprocessing': 'CoM -> Deskew', 'features': 'Pixel 20% + HOG 80%',
                   'k': 3, 'weights': 'distance', 'distance': 'euclidean',
                   'handmade_aug': 'EXP007 size_jitter + stroke_jitter; train writer/class balanced',
                   'stroke_strength': AUG_STRENGTH, 'seed': SEED},
        'normal_reference': normal_result,
        'normal_official_validation': reference_val,
        'candidate_oof': aggregates,
        'candidate_results': summary_rows,
        'fold_results': metrics_by_fold,
        'augmentation_details': augmentation_records,
        'development_candidate': best['variant'],
        'official_guardrail_min_accuracy': MIN_OFFICIAL_VALIDATION_ACCURACY,
        'official_test_used': False,
        'warning': 'All four handwriting writers were used in earlier Normal model selection. Writer-held-out fit is applied in these folds but this is repeated-use development OOF, NOT an independent writer test.',
        'elapsed_seconds': round(time.perf_counter()-t0, 2),
        'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                        'sklearn': sklearn.__version__},
    }
    (OUT / 'metrics.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nEXP-CMB-003 SUMMARY (4-writer development OOF)', flush=True)
    for row in summary_rows:
        print(f"{row['variant']:17s} OOF={row['accuracy']:.6f} F1={row['macro_f1']:.6f} "
              f"official_min={row['min_fold_official_accuracy']:.6f} "
              f"guardrail={'PASS' if row['official_guardrail_pass'] else 'FAIL'}", flush=True)
    print('SELECTED DEVELOPMENT CANDIDATE:', best['variant'], flush=True)
    print('Saved:', OUT, flush=True)
    print('Official Test was not accessed.', flush=True)


if __name__ == '__main__':
    main()


## EXP_CMB_004_KNN_Combined_Sample_Influence

원본: [05_model_optimize/KNN/Combined/scripts/EXP_CMB_004_KNN_Combined_Sample_Influence.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/scripts/EXP_CMB_004_KNN_Combined_Sample_Influence.py)  
Git blob: `1b8445dbbfb043a10f050f6352d12647f640c82c`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/logs)

EXP-CMB-004: training-writer sample duplication, 4-writer development OOF.

Place in 05_model_optimize/KNN/Combined/scripts/ and run from the repo root.
Prerequisites: existing Normal scripts and EXP_CMB_001/002/003 scripts.
Official Test is never accessed. No held-out writer samples enter model fit.
Duplicates are intentionally exact; with k=3 they may monopolize the vote.



In [ ]:
%%archive_source 05_model_optimize/KNN/Combined/scripts/EXP_CMB_004_KNN_Combined_Sample_Influence.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-CMB-004: training-writer sample duplication, 4-writer development OOF.

Place in 05_model_optimize/KNN/Combined/scripts/ and run from the repo root.
Prerequisites: existing Normal scripts and EXP_CMB_001/002/003 scripts.
Official Test is never accessed. No held-out writer samples enter model fit.
Duplicates are intentionally exact; with k=3 they may monopolize the vote.
"""
from __future__ import annotations

import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn

ROOT = Path(__file__).resolve().parents[4]
NORMAL_DIR = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
COMBINED_DIR = ROOT / '05_model_optimize' / 'KNN' / 'Combined' / 'scripts'
for folder in (NORMAL_DIR, COMBINED_DIR, ROOT / '03_experiments' / 'model_optimization'):
    sys.path.insert(0, str(folder))

from common import LABELS, fixed_split
from EXP004_KNN_Normal_CoM import BatchedCoM
from final_knn_normal import assemble_training, prepare_features, EXPECTED_VALIDATION_CORRECT
from EXP_CMB_001_KNN_Combined_Baseline import WRITERS, load_writer, save_csv, save_cm, summary
from EXP_CMB_002_KNN_Combined_Optimized import fresh_knn, paired_counts
from EXP_CMB_003_KNN_Combined_Handmade_Augmentation import (
    SEED as AUG_SEED, balanced_nested_indices, make_aug_features,
)

OUT = ROOT / '05_model_optimize' / 'KNN' / 'Combined' / 'results' / 'EXP_CMB_004_sample_influence'
VARIANTS = ('control', 'repeat_original_50', 'repeat_augmented_50', 'repeat_mixed_50')
EXPECTED_CONTROL_BY_WRITER = {41: 587, 42: 596, 43: 621, 44: 597}
EXPECTED_CONTROL_OOF_CORRECT = 2401
MIN_OFFICIAL_VALIDATION_ACCURACY = 0.978


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    xtr, ytr, xva, yva, official_sha = fixed_split()
    if (len(ytr), len(yva)) != (76800, 19200):
        raise RuntimeError('Official Training/Validation split differs from baseline')
    writers = {w: load_writer(w) for w in WRITERS}
    com = BatchedCoM()
    print('Official Test NOT loaded. Frozen Normal feature pipeline...', flush=True)
    official_x, official_y = assemble_training(xtr, ytr, com)
    del xtr, ytr
    if official_x.shape != (107520, 1108):
        raise RuntimeError('Official-derived feature shape mismatch')
    official_val_x = prepare_features(xva, com, 'official Validation')
    del xva
    writer_features = {w: prepare_features(writers[w][0], com, f'writer {w}') for w in WRITERS}

    # Reproduce the frozen Normal official Validation before investigating variants.
    normal = fresh_knn()
    normal.fit(official_x, official_y)
    official_control = summary(yva, normal.predict(official_val_x))
    if official_control['correct'] != EXPECTED_VALIDATION_CORRECT:
        raise RuntimeError('Frozen Normal official Validation control was not reproduced')
    del normal

    predictions_by_variant = {name: {} for name in VARIANTS}
    fold_results = []
    records = []
    for heldout in WRITERS:
        train_writers = [w for w in WRITERS if w != heldout]
        if heldout in train_writers or len(train_writers) != 3:
            raise RuntimeError('Held-out writer data leakage')
        raw_x = np.concatenate([writers[w][0] for w in train_writers])
        raw_y = np.concatenate([writers[w][1] for w in train_writers])
        ids = np.concatenate([np.full(640, w, dtype=np.int32) for w in train_writers])
        original_x = np.concatenate([writer_features[w] for w in train_writers])
        if raw_y.shape != (1920,) or original_x.shape != (1920, 1108):
            raise RuntimeError('Hand-made training shape mismatch')
        # Use EXACT nested sampling and deterministic augmented rows from EXP-CMB-003.
        selected = balanced_nested_indices(ids, raw_y, seed=AUG_SEED + heldout)
        all_indices = selected[100]
        aug_x, aug_info = make_aug_features(raw_x, all_indices, com, 100, heldout)
        aug_y = raw_y[all_indices]
        base_x = np.concatenate((official_x, original_x, aug_x), axis=0)
        base_y = np.concatenate((official_y, raw_y, aug_y), axis=0)
        if base_x.shape != (111360, 1108) or base_y.shape != (111360,):
            raise RuntimeError('EXP-CMB-003 100% reference training shape mismatch')

        # 20 / class / writer = 960 samples. Mixed is 10 original and 10 augmented
        # per writer/class, hence all three variants add an equal 960 rows.
        original_50 = selected[50]
        augmented_50 = selected[50]
        original_25 = selected[25]
        augmented_other_25 = np.setdiff1d(selected[50], selected[25], assume_unique=True)
        if (len(original_50), len(augmented_50), len(original_25), len(augmented_other_25)) != (960, 960, 480, 480):
            raise RuntimeError('Invalid balanced sample counts')
        extra_parts = {
            'control': (None, None),
            'repeat_original_50': (original_x[original_50], raw_y[original_50]),
            'repeat_augmented_50': (aug_x[augmented_50], aug_y[augmented_50]),
            'repeat_mixed_50': (
                np.concatenate((original_x[original_25], aug_x[augmented_other_25])),
                np.concatenate((raw_y[original_25], aug_y[augmented_other_25])),
            ),
        }
        print(f'\n[Fold heldout={heldout}] train writers={train_writers}; control rows=111360', flush=True)
        for variant in VARIANTS:
            extra_x, extra_y = extra_parts[variant]
            fit_x = base_x if extra_x is None else np.concatenate((base_x, extra_x), axis=0)
            fit_y = base_y if extra_y is None else np.concatenate((base_y, extra_y), axis=0)
            expected_rows = 111360 if variant == 'control' else 112320
            if fit_x.shape != (expected_rows, 1108) or fit_y.shape != (expected_rows,):
                raise RuntimeError('Training row count mismatch')
            model = fresh_knn()
            t0 = time.perf_counter()
            model.fit(fit_x, fit_y)
            fit_seconds = time.perf_counter() - t0
            t0 = time.perf_counter()
            pred_hand = model.predict(writer_features[heldout]).astype(np.int64)
            predict_hand_s = time.perf_counter() - t0
            t0 = time.perf_counter()
            pred_official = model.predict(official_val_x).astype(np.int64)
            predict_official_s = time.perf_counter() - t0
            del model
            hand_stat = summary(writers[heldout][1], pred_hand)
            official_stat = summary(yva, pred_official)
            if variant == 'control' and hand_stat['correct'] != EXPECTED_CONTROL_BY_WRITER[heldout]:
                raise RuntimeError(f'EXP-CMB-003 control mismatch for heldout={heldout}: '
                                   f"{hand_stat['correct']} != {EXPECTED_CONTROL_BY_WRITER[heldout]}")
            predictions_by_variant[variant][heldout] = pred_hand
            row = {
                'variant': variant, 'heldout_writer': heldout,
                'train_writers': ','.join(map(str, train_writers)),
                'official_derived': 107520, 'handmade_original': 1920,
                'handmade_augmented': 1920, 'repeated_rows': expected_rows - 111360,
                'train_rows': expected_rows, 'handmade_correct': hand_stat['correct'],
                'handmade_accuracy': hand_stat['accuracy'], 'handmade_macro_f1': hand_stat['macro_f1'],
                'official_correct': official_stat['correct'],
                'official_accuracy': official_stat['accuracy'],
                'official_macro_f1': official_stat['macro_f1'],
                'official_guardrail_pass': int(official_stat['accuracy'] >= MIN_OFFICIAL_VALIDATION_ACCURACY),
                'fit_seconds': round(fit_seconds, 3),
                'handmade_predict_seconds': round(predict_hand_s, 3),
                'official_predict_seconds': round(predict_official_s, 3),
            }
            fold_results.append(row)
            save_csv(OUT / 'fold_results_partial.csv', fold_results)
            print(f'  {variant:21s} train={expected_rows} | OOF_fold={hand_stat["accuracy"]:.6f} '
                  f'Official={official_stat["accuracy"]:.6f} | guardrail='
                  f'{"PASS" if row["official_guardrail_pass"] else "FAIL"}', flush=True)
        records.append({'heldout_writer': heldout, 'train_writers': train_writers,
                        'augmentation': aug_info, 'repeat_count': 960})
        del base_x, base_y, raw_x, raw_y, aug_x, aug_y, original_x, extra_parts

    truth = np.concatenate([writers[w][1] for w in WRITERS])
    control_pred = np.concatenate([predictions_by_variant['control'][w] for w in WRITERS])
    if int((control_pred == truth).sum()) != EXPECTED_CONTROL_OOF_CORRECT:
        raise RuntimeError('EXP-CMB-003 100% OOF summary did not reproduce')
    result_rows = []
    aggregate = {}
    all_predictions = []
    for variant in VARIANTS:
        pred = np.concatenate([predictions_by_variant[variant][w] for w in WRITERS])
        s = summary(truth, pred)
        paired = paired_counts(truth, control_pred, pred)
        official_min = min(row['official_accuracy'] for row in fold_results if row['variant'] == variant)
        result_rows.append({
            'variant': variant, 'oof_correct': s['correct'], 'oof_accuracy': s['accuracy'],
            'oof_macro_f1': s['macro_f1'], 'delta_pp_vs_control':
                100.0 * (s['accuracy'] - EXPECTED_CONTROL_OOF_CORRECT / 2560),
            'min_official_accuracy': official_min,
            'official_guardrail_pass': int(official_min >= MIN_OFFICIAL_VALIDATION_ACCURACY),
        })
        aggregate[variant] = {'oof': s, 'paired_vs_control': paired}
        save_cm(OUT / f'{variant}_oof_confusion_matrix.csv', truth, pred)
        save_csv(OUT / f'{variant}_per_class_oof.csv', [
            {'class': label, **s['per_class'][label]} for label in LABELS
        ])
        for offset, w in enumerate(WRITERS):
            for i in range(640):
                j = offset * 640 + i
                all_predictions.append({
                    'variant': variant, 'heldout_writer': w, 'index': i,
                    'true': LABELS[int(truth[j])], 'predicted': LABELS[int(pred[j])],
                    'correct': int(pred[j] == truth[j]),
                })
    eligible = [row for row in result_rows if row['official_guardrail_pass']]
    if not eligible:
        raise RuntimeError('No candidate passed the prespecified official guardrail')
    selected = max(eligible, key=lambda r: (r['oof_macro_f1'], r['oof_accuracy'],
                                            -int(r['variant'] != 'control')))
    save_csv(OUT / 'candidate_results.csv', result_rows)
    save_csv(OUT / 'fold_results.csv', fold_results)
    save_csv(OUT / 'official_validation.csv', [
        {k: row[k] for k in ('variant', 'heldout_writer', 'official_correct', 'official_accuracy',
                            'official_macro_f1')} for row in fold_results
    ])
    save_csv(OUT / 'oof_predictions.csv', all_predictions)
    report = {
        'experiment': 'EXP-CMB-004 Hand-made sample duplication influence (four-writer development OOF)',
        'official_dataset_sha256': official_sha,
        'handmade_sha256': {str(w): writers[w][2] for w in WRITERS},
        'control_reference': 'EXP-CMB-003 handmade_aug_100',
        'training_rows': {'control': 111360, 'duplicated_variants': 112320},
        'variants': list(VARIANTS),
        'method': '960 existing Hand-made source rows duplicated exactly; equal count across 3 variants; '
                  'writer/class balanced; k=3 distance Euclidean unchanged',
        'fold_results': fold_results,
        'candidate_results': result_rows,
        'aggregate': aggregate,
        'selected_development_candidate': selected['variant'],
        'augmentation_records': records,
        'official_guardrail_min_accuracy': MIN_OFFICIAL_VALIDATION_ACCURACY,
        'official_test_used': False,
        'warning': 'This is repeated-use DEVELOPMENT writer-held-out OOF, NOT a truly independent test. '
                   'Exact duplication can fill k=3 neighbor slots and alter majority voting.',
        'elapsed_seconds': round(time.perf_counter() - started, 2),
        'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                        'sklearn': sklearn.__version__},
    }
    (OUT / 'metrics.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
    print('\nEXP-CMB-004 SUMMARY (development 4-writer OOF)', flush=True)
    for row in result_rows:
        print(f"{row['variant']:21s} OOF={row['oof_accuracy']:.6f} "
              f"F1={row['oof_macro_f1']:.6f} official_min={row['min_official_accuracy']:.6f} "
              f"guardrail={'PASS' if row['official_guardrail_pass'] else 'FAIL'}", flush=True)
    print('SELECTED DEVELOPMENT CANDIDATE:', selected['variant'], flush=True)
    print('Saved:', OUT, flush=True)
    print('Official Test was not accessed.', flush=True)


if __name__ == '__main__':
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")